# Data Description
- Background: Unit 1.7 補充資料驗證：號誌路口位置。提供機關：臺北市交通局交通管制工程處；官方來源：https://data.taipei/dataset/detail?id=4a599738-6550-448a-9a78-03b26c67e249。原始來源對應與官方最新資源bytes完全相同為不同命題；除歷史界內包已逐byte雜湊對應外，未重新下載認證其他最新資源。
- date received: 學生於本單元提供；實際下載／收到日期未提供。首次本機檢視2026-11-26，不將今日當下載日。
- Path to data file: 官方來源及版本如上；精確本機容器如下，解壓至data/temp僅為衍生暫存。
  - `data/raw/號誌路口經緯度一覽20230706.csv`；150,844 bytes；SHA-256 `4de42a369f6b6ba38df29ff2d357f8ad5aec7a237b8709edc431d77c3cfb29a7`。
- Unit of observation: 號誌路口位置紀錄，非燈頭或設備。
- Sample period: 檔名20230706非每筆觀測／設置時間；實際快照時點仍未認證；發布／更新：資源更新2025-07-15。ZIP時間、檔名時間、觀測／設置時間分開，不由檔名猜全年存量。
- Known issues: 原檔為tab分隔；路口位置紀錄不等於燈頭或設備數，完整歷史號誌存量未證。缺失、重複、坐標及CRS由下方實讀輸出；沒有完整性證據的0僅為本檔未觀測。授權：政府資料開放授權條款第1版；依官方頁，來源與版本須保留。
- Definition for each variable: 下方逐欄型態、缺失、觀測域與定義表。已知官方欄意與本次診斷欄分開；未獲欄位碼表支持者明記未知，不依名稱猜設施單位。
- Data path: 上列每個獨立原始容器；實際讀取成員列於Summary Stats。
- Role: raw。
- Verification status: 以本Notebook乾淨kernel實際輸出與執行metadata為準；不認證已清理。
- SHA-256: 上列每檔固定綁定，載入前逐項重核。
- Next processing notebook: 本單元僅診斷，Chapter 2尚未建立；[驗證一](../code/taipei_supplementary_accident_boundary_validation.ipynb)、[驗證二](../code/taipei_transportation_infrastructure_validation.ipynb)。

Author: 詹明翰、謝子詮、曾翊涵；Created At / Last Modified At: 2026-11-26。

Primary Owner：謝子詮（DE）；Collaborator / Analytical Reviewer：詹明翰（DA）；Project Reviewer：曾翊涵（PM）。共同作者與職務不代表已完成個別工作。本輪由Codex協助執行，成員驗收與審查尚待完成。


In [1]:
from pathlib import Path
import sys, hashlib, json, zipfile
import pandas as pd
from IPython.display import display, Markdown
for candidate in (Path.cwd().resolve(), *Path.cwd().resolve().parents):
    if (candidate/'AGENTS.md').is_file() and (candidate/'templates').is_dir():
        PROJECT_ROOT=candidate;break
else:raise FileNotFoundError('請從本研究工作包內執行')
def fingerprint(path):
    h=hashlib.sha256()
    with path.open('rb') as f:
        for b in iter(lambda:f.read(4*1024*1024),b''):h.update(b)
    return h.hexdigest()
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_config.json')=='7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7'
assert fingerprint(PROJECT_ROOT/'data/code/unit17_validation_support.py')=='1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080'
sys.path.insert(0,str(PROJECT_ROOT/'data/code'))
from unit17_validation_support import Context, show, safe_text
pd.set_option('display.max_columns',None)
pd.set_option('display.max_colwidth',160)
ctx=Context(PROJECT_ROOT,'7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7')
print('Python:',sys.version,'Config:','7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7','Helper:','1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080')


Python: 3.12.2 (tags/v3.12.2:6abddd9, Feb  6 2024, 21:26:36) [MSC v.1937 64 bit (AMD64)] Config: 7554b54042b43dd8fd2ec2c5cfa6f218e9c6acae2376ff7e0aee0e14e73a17a7 Helper: 1a18dd248daf08074a5ba7ec659a489fe7b17852195259e8084727d4fd6f3080


## Change for Your Own Data

使用已核對精確路徑及固定SHA-256；不選第一個glob或最新檔。CP950非法bytes只作可逆診斷；大檔分批讀取、OSM僅研究區2km bbox，保留全國母體數。

In [2]:
FAMILY='signals'
INPUTS=ctx.family_items(FAMILY)
for item in INPUTS:ctx.file(item['path'])
frames=ctx.load_family(FAMILY)

## Summary

每個容器、實際成員與載入範圍分列；沒有將暫存解壓檔當原始下載資料。

In [3]:
show('固定輸入身分',pd.DataFrame([{k:v for k,v in r.items() if k not in ['members','encoding_checks']} for r in INPUTS]))
show('讀取範圍與CRS／編碼',pd.DataFrame(ctx.summaries[FAMILY]))

**固定輸入身分**

,family,path,bytes,sha256,status
0,signals,data/raw/號誌路口經緯度一覽20230706.csv,150844,4de42a369f6b6ba38df29ff2d357f8ad5aec7a237b8709edc431d77c3cfb29a7,present


**讀取範圍與CRS／編碼**

,path,rows,encoding,strict,byte_roundtrip,delimiter,invalid_byte_count,invalid_bytes
0,data/raw/號誌路口經緯度一覽20230706.csv,2710,cp950,True,True,'\t',0,[]


### Sample

各實際讀取表前十列；全國事故為臺北篩選後第一當事人，不代表全國前十列。OSM為bbox空間子集。

In [4]:
for source,frame in frames.items():
    show(source,frame.drop(columns='geometry',errors='ignore').head(10),block=10)

**data/raw/號誌路口經緯度一覽20230706.csv**

,流水號,地點,行政區,WGS經度座標,WGS緯度座標,_source_row,_source_path,_decoding_valid
0,1,老泉街 防汛道路,文山區,121.55754,24.976234,1,data/raw/號誌路口經緯度一覽20230706.csv,True
1,2,木新路三段 232巷,文山區,121.55911,24.981323,2,data/raw/號誌路口經緯度一覽20230706.csv,True
2,3,木新路三段 寶橋路,文山區,121.554054,24.98002,3,data/raw/號誌路口經緯度一覽20230706.csv,True
3,4,老泉街 恆光橋頭,文山區,121.56776,24.979765,4,data/raw/號誌路口經緯度一覽20230706.csv,True
4,5,木新路三段 一壽街,文山區,121.55637,24.980597,5,data/raw/號誌路口經緯度一覽20230706.csv,True
5,6,木新路三段 310巷,文山區,121.55716,24.98081,6,data/raw/號誌路口經緯度一覽20230706.csv,True
6,7,興隆路四段 122,文山區,121.56234,24.982647,7,data/raw/號誌路口經緯度一覽20230706.csv,True
7,8,保儀路13巷 26巷,文山區,121.56861,24.988333,8,data/raw/號誌路口經緯度一覽20230706.csv,True
8,9,恆光街 恆光橋,文山區,121.567024,24.9825,9,data/raw/號誌路口經緯度一覽20230706.csv,True
9,10,木新路三段 278巷,文山區,121.55807,24.981054,10,data/raw/號誌路口經緯度一覽20230706.csv,True


### Summary Stats

ZIP逐成員CRC、配套、原始WKT、全部欄型態／缺失／定義、重複與數值／類別／時間診斷。保留原值及bytes。

In [5]:
_ = ctx.raw_report(FAMILY,crc=True)

,source_metadata_and_limits
title,號誌路口位置
provider,臺北市交通局交通管制工程處
url,https://data.taipei/dataset/detail?id=4a599738-6550-448a-9a78-03b26c67e249
publication,資源更新2025-07-15
period,檔名20230706非每筆觀測／設置時間；實際快照時點仍未認證
limitation,原檔為tab分隔；路口位置紀錄不等於燈頭或設備數，完整歷史號誌存量未證
license,政府資料開放授權條款第1版；依官方頁，來源與版本須保留


,family,path,bytes,sha256,status
0,signals,data/raw/號誌路口經緯度一覽20230706.csv,150844,4de42a369f6b6ba38df29ff2d357f8ad5aec7a237b8709edc431d77c3cfb29a7,present


**實際讀取範圍／編碼／CRS／全國或地區計數**

,path,rows,encoding,strict,byte_roundtrip,delimiter,invalid_byte_count,invalid_bytes
0,data/raw/號誌路口經緯度一覽20230706.csv,2710,cp950,True,True,'\t',0,[]


### data/raw/號誌路口經緯度一覽20230706.csv

**實際前十列；全國資料表為篩選後臺北第一當事人，非全國母體**

... (6 outputs left out)

,index,count,mean,std,min,25%,50%,75%,max
0,WGS經度座標,2699.0,121.49903,2.339787,0.0,121.517470,121.541300,121.567988,121.62302
1,WGS緯度座標,2699.0,25.04728,0.483597,0.0,25.033876,25.052303,25.077925,25.16750


,field,missing_or_unparseable,zero
0,WGS經度座標,11,1
1,WGS緯度座標,11,1


原始屬性完整列重複（不含診斷追溯欄）

 0


本文件只確認資料身分及可讀性；語義／日期不足者限制用途，不強行納入正式Dataset B3。人類驗收仍待完成。

In [6]:
for item in INPUTS:ctx.file(item['path'])
print('原始容器hash重核通過；正式分析資料輸出0。')

原始容器hash重核通過；正式分析資料輸出0。
